# CineGraph: MovieLens Exploratory Data Analysis & Distributed Spark Processing
This notebook demonstrates distributed data exploration, Spark SQL aggregations, and collaborative filtering metrics on the MovieLens dataset.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set PySpark environment variables
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName('CineGraph-EDA') \
    .master('local[*]') \
    .config('spark.driver.memory', '2g') \
    .config('spark.driver.bindAddress', '127.0.0.1') \
    .getOrCreate()

print('Spark Version:', spark.version)

## 1. Load Processed MovieLens Dataset into Spark DataFrames

In [ ]:
ratings_path = '../data/processed/ratings.csv'
movies_path = '../data/processed/movies.csv'

ratings_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(ratings_path)
movies_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(movies_path)

ratings_df.createOrReplaceTempView('ratings')
movies_df.createOrReplaceTempView('movies')

print(f'Total Ratings: {ratings_df.count()}, Total Movies: {movies_df.count()}')
ratings_df.show(5)

## 2. Top 10 Most Popular Movies via Spark SQL

In [ ]:
top_popular = spark.sql('''
    SELECT m.movieId, m.title, m.genres, 
           COUNT(r.rating) AS rating_count,
           ROUND(AVG(r.rating), 2) AS avg_rating
    FROM ratings r
    JOIN movies m ON r.movieId = m.movieId
    GROUP BY m.movieId, m.title, m.genres
    ORDER BY rating_count DESC
    LIMIT 10
''')
top_popular.show(truncate=False)

## 3. Ratings Distribution Analysis

In [ ]:
rating_dist = ratings_df.groupBy('rating').count().orderBy('rating').toPandas()

plt.figure(figsize=(8, 4))
sns.barplot(data=rating_dist, x='rating', y='count', palette='viridis')
plt.title('MovieLens Rating Volume Distribution')
plt.xlabel('Star Rating')
plt.ylabel('Number of Reviews')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 4. Genre Breakdown & Analysis

In [ ]:
genre_df = movies_df \
    .withColumn('genre', F.explode(F.split(F.col('genres'), r'\|'))) \
    .filter((F.col('genre') != '') & (F.col('genre') != 'Other')) \
    .groupBy('genre') \
    .agg(F.count('movieId').alias('movie_count')) \
    .orderBy(F.col('movie_count').desc()) \
    .toPandas()

plt.figure(figsize=(10, 5))
sns.barplot(data=genre_df, x='movie_count', y='genre', palette='mako')
plt.title('Movie Count per Genre in MovieLens')
plt.xlabel('Number of Titles')
plt.show()

## 5. Clean Shutdown

In [ ]:
spark.stop()
print('Spark session successfully stopped.')